<a href="https://colab.research.google.com/github/Celso-RQ-Valle/Modeling-Functions/blob/main/Metrics_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
import pandas as pd
from scipy.stats import ks_2samp
from itertools import combinations
from pyspark.sql import functions as F

In [26]:
def calculate_ks(y_true, y_pred):
    if y_true.empty or y_pred.empty or y_true.sum() == 0 or y_pred.sum() == 0:
        return float('nan')

    return ks_2samp(
        y_pred[y_true == 1],
        y_pred[y_true == 0]
    ).statistic

def ks(score_column, df_pd, grupo, target):

    if not grupo:
        ks_value = calculate_ks(df_pd[target], df_pd[score_column])
        return pd.DataFrame({"KS": [ks_value]})

    df_grouped = (
        df_pd
        .groupby(grupo)
        .apply(lambda x: calculate_ks(x[target], x[score_column]))
        .reset_index(name="KS")
    )

    return df_grouped

from itertools import combinations
from pyspark.sql import functions as F


def calcula_ks_por_combinacao(df_spark, grupos, scores, target):
    """
    Calcula KS para todas combinações de grupos e lista de scores.
    Assume que já existe uma função ks(score, df_pd, grupo, target).
    """

    # Gera todas combinações possíveis dos grupos
    grupos_combinacoes = [[]] + [
      list(c)
      for i in range(1, len(grupos) + 1)
      for c in combinations(grupos, i)
]

    df_KS = None

    # Converte Spark → Pandas uma única vez
    df_pd = df_spark.toPandas()

    for score in scores:

        for comb in grupos_combinacoes:

            diff = [g for g in grupos if g not in comb]

            df_ks_aux = (
                spark.createDataFrame(
                    ks(score, df_pd, grupo=comb, target=target)
                )
                .withColumn("score", F.lit(score))
            )

            # Cast colunas existentes para string
            for g in grupos:
                if g in df_ks_aux.columns:
                    df_ks_aux = df_ks_aux.withColumn(g, F.col(g).cast("string"))

            # Adiciona colunas faltantes como "Geral"
            for col in diff:
                df_ks_aux = df_ks_aux.withColumn(col, F.lit("Geral"))

            # Garante todas as colunas de grupo
            for g in grupos:
                if g not in df_ks_aux.columns:
                    df_ks_aux = df_ks_aux.withColumn(g, F.lit("Geral"))

            # Reordena
            select_cols = grupos + ["KS", "score"]
            df_ks_aux = df_ks_aux.select(select_cols)

            if df_KS is None:
                df_KS = df_ks_aux
            else:
                df_KS = df_KS.unionByName(df_ks_aux, allowMissingColumns=True)

    return df_KS

In [42]:
from itertools import combinations
from pyspark.sql import functions as F


def calculate_metrics(df_spark, grupos, scores, target):
    """
    Calcula KS, AUC e Gini para todas combinações de grupos e lista de scores.
    Assume que já existem funções:
        ks(score, df_pd, grupo, target)
        auc_gini(score, df_pd, grupo, target)
    """

    # Gera todas combinações possíveis incluindo Geral
    grupos_combinacoes = [[]] + [
        list(c)
        for i in range(1, len(grupos) + 1)
        for c in combinations(grupos, i)
    ]

    df_metrics = None

    # Converte Spark → Pandas uma única vez
    df_pd = df_spark.toPandas()

    for score in scores:

        for comb in grupos_combinacoes:

            diff = [g for g in grupos if g not in comb]

            #  Calcula KS
            df_ks = ks(score, df_pd, grupo=comb, target=target)

            #  Calcula AUC e Gini
            df_auc = auc_gini(score, df_pd, grupo=comb, target=target)

            #  Merge pandas (garante mesmas chaves de grupo)
            if comb:
                df_merge = df_ks.merge(df_auc, on=comb, how="left")
            else:
                df_merge = df_ks.copy()
                df_merge["AUC"] = df_auc["AUC"]
                df_merge["Gini"] = df_auc["Gini"]

            # Converte para Spark
            df_aux = (
                spark.createDataFrame(df_merge)
                .withColumn("score", F.lit(score))
            )
            # Arredonda métricas para 5 casas
            df_aux = (
                df_aux
                .withColumn("KS", F.round(F.col("KS"), 5))
                .withColumn("AUC", F.round(F.col("AUC"), 5))
                .withColumn("Gini", F.round(F.col("Gini"), 5))
            )

            # Cast colunas existentes
            for g in grupos:
                if g in df_aux.columns:
                    df_aux = df_aux.withColumn(g, F.col(g).cast("string"))

            # Adiciona colunas faltantes como Geral
            for col in diff:
                df_aux = df_aux.withColumn(col, F.lit("Geral"))

            for g in grupos:
                if g not in df_aux.columns:
                    df_aux = df_aux.withColumn(g, F.lit("Geral"))

            # Reordena
            select_cols = grupos + ["KS", "AUC", "Gini", "score"]
            df_aux = df_aux.select(select_cols)

            if df_metrics is None:
                df_metrics = df_aux
            else:
                df_metrics = df_metrics.unionByName(
                    df_aux,
                    allowMissingColumns=True
                )

    return df_metrics

In [40]:
import pandas as pd
from sklearn.metrics import roc_auc_score


def calculate_auc_gini(y_true, y_pred):
    """
    Calcula AUC e Gini com proteções contra casos degenerados.
    """

    if (
        y_true.empty
        or y_pred.empty
        or y_true.nunique() < 2  # precisa ter classe 0 e 1
    ):
        return float('nan'), float('nan')

    auc = roc_auc_score(y_true, y_pred)
    gini = 2 * auc - 1

    return auc, gini


def auc_gini(score_column, df_pd, grupo, target):
    """
    Calcula AUC e Gini por grupo.
    Mesma lógica estrutural da função ks().
    """

    # Caso Geral
    if not grupo:
        auc, gini = calculate_auc_gini(
            df_pd[target],
            df_pd[score_column]
        )

        return pd.DataFrame({
            "AUC": [auc],
            "Gini": [gini]
        })

    # Quebra por grupo
    def _calc(x):
        auc, gini = calculate_auc_gini(
            x[target],
            x[score_column]
        )
        return pd.Series({
            "AUC": auc,
            "Gini": gini
        })

    df_grouped = (
        df_pd
        .groupby(grupo)
        .apply(_calc)
        .reset_index()
    )

    return df_grouped

In [44]:
grupos = ['safra', 'proposta', 'flag_grupo1', 'flag_grupo2']
#grupos = ['safra']
scores = ['score1', 'score2', 'score3']

df_KS = calculate_metrics(
    df_spark=df,
    grupos=grupos,
    scores=scores,
    target="target"
)

df_KS.show(5000)

/tmp/ipython-input-449419695.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: calculate_ks(x[target], x[score_column]))
/tmp/ipython-input-2713901335.py:55: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(_calc)
/tmp/ipython-input-449419695.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pan

+-------+--------+-----------+-----------+-------+-------+--------+------+
|  safra|proposta|flag_grupo1|flag_grupo2|     KS|    AUC|    Gini| score|
+-------+--------+-----------+-----------+-------+-------+--------+------+
|  Geral|   Geral|      Geral|      Geral|0.09862|0.56452| 0.12904|score1|
|2023-01|   Geral|      Geral|      Geral|0.12378|0.57579| 0.15158|score1|
|2023-02|   Geral|      Geral|      Geral|0.11431|  0.571|   0.142|score1|
|2023-03|   Geral|      Geral|      Geral| 0.0903|0.55542| 0.11084|score1|
|2023-04|   Geral|      Geral|      Geral|0.08758| 0.5423|  0.0846|score1|
|  Geral| digital|      Geral|      Geral|0.08955|0.55608| 0.11216|score1|
|  Geral|    loja|      Geral|      Geral|0.06426|0.54018| 0.08036|score1|
|  Geral|parceiro|      Geral|      Geral|0.07994| 0.5358|  0.0716|score1|
|  Geral|   Geral|          0|      Geral|0.08774|0.55376| 0.10752|score1|
|  Geral|   Geral|          1|      Geral|0.08548|0.55267| 0.10535|score1|
|  Geral|   Geral|      G

## artificial DF Pd

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

# Tamanho da base
n = 20000

# ----------------------------
# Grupos
# ----------------------------

grupos = ['safra', 'proposta', 'flag_grupo1', 'flag_grupo2']
scores = ['score1', 'score2', 'score3']

df = pd.DataFrame({
    'safra': np.random.choice(
        ['2023-01', '2023-02', '2023-03', '2023-04'],
        size=n,
        p=[0.2, 0.3, 0.3, 0.2]
    ),
    'proposta': np.random.choice(
        ['digital', 'loja', 'parceiro'],
        size=n,
        p=[0.5, 0.3, 0.2]
    ),
    'flag_grupo1': np.random.binomial(1, 0.4, size=n),
    'flag_grupo2': np.random.binomial(1, 0.3, size=n)
})

# ----------------------------
# Criando risco base
# ----------------------------

# risco base por proposta
risco_proposta = {
    'digital': 0.08,
    'loja': 0.12,
    'parceiro': 0.18
}

# risco adicional por flag
df['prob_base'] = (
    df['proposta'].map(risco_proposta) +
    df['flag_grupo1'] * 0.05 +
    df['flag_grupo2'] * 0.07 +
    np.random.normal(0, 0.01, n)  # ruído
)

# limitando probabilidade
df['prob_base'] = df['prob_base'].clip(0.01, 0.8)

# ----------------------------
# Target binária
# ----------------------------

df['target'] = np.random.binomial(1, df['prob_base'])

# ----------------------------
# Scores simulados
# ----------------------------

# score1 → bom modelo
df['score1'] = (
    0.7 * df['prob_base'] +
    np.random.normal(0, 0.05, n)
)

# score2 → modelo médio
df['score2'] = (
    0.4 * df['prob_base'] +
    np.random.normal(0, 0.10, n)
)

# score3 → modelo fraco
df['score3'] = np.random.uniform(0, 1, n)

# Normalizando scores entre 0 e 1
for s in scores:
    df[s] = (df[s] - df[s].min()) / (df[s].max() - df[s].min())

df.head()

,safra,proposta,flag_grupo1,flag_grupo2,prob_base,target,score1,score2,score3
0,2023-02,loja,0,1,0.186734,0,0.445340,0.607451,0.014455
1,2023-04,digital,0,1,0.152929,0,0.505658,0.484047,0.807917
2,2023-03,digital,0,0,0.077754,0,0.301285,0.657362,0.976237
3,2023-03,loja,0,0,0.124389,1,0.455621,0.397452,0.674111
4,2023-01,digital,0,0,0.076233,0,0.427009,0.382637,0.821436


## Artifical DF - Spark

In [2]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# ----------------------------
# Parâmetros
# ----------------------------

n = 20000

grupos = ['safra', 'proposta', 'flag_grupo1', 'flag_grupo2']
scores = ['score1', 'score2', 'score3']

# ----------------------------
# Base inicial
# ----------------------------

df = spark.range(n)

# ----------------------------
# Variáveis categóricas
# ----------------------------

df = (
    df
    .withColumn(
        "safra",
        F.when(F.rand() < 0.2, "2023-01")
         .when(F.rand() < 0.5, "2023-02")
         .when(F.rand() < 0.8, "2023-03")
         .otherwise("2023-04")
    )
    .withColumn(
        "proposta",
        F.when(F.rand() < 0.5, "digital")
         .when(F.rand() < 0.8, "loja")
         .otherwise("parceiro")
    )
    .withColumn("flag_grupo1", (F.rand() < 0.4).cast("int"))
    .withColumn("flag_grupo2", (F.rand() < 0.3).cast("int"))
)

# ----------------------------
# Probabilidade base
# ----------------------------

df = (
    df
    .withColumn(
        "risco_proposta",
        F.when(F.col("proposta") == "digital", 0.08)
         .when(F.col("proposta") == "loja", 0.12)
         .otherwise(0.18)
    )
    .withColumn(
        "prob_base",
        F.col("risco_proposta") +
        F.col("flag_grupo1") * 0.05 +
        F.col("flag_grupo2") * 0.07 +
        F.randn() * 0.01
    )
    .withColumn(
        "prob_base",
        F.when(F.col("prob_base") < 0.01, 0.01)
         .when(F.col("prob_base") > 0.8, 0.8)
         .otherwise(F.col("prob_base"))
    )
)

# ----------------------------
# Target binária
# ----------------------------

df = df.withColumn(
    "target",
    (F.rand() < F.col("prob_base")).cast("int")
)

# ----------------------------
# Scores simulados
# ----------------------------

df = (
    df
    # score bom
    .withColumn(
        "score1",
        0.7 * F.col("prob_base") + F.randn() * 0.05
    )
    # score médio
    .withColumn(
        "score2",
        0.4 * F.col("prob_base") + F.randn() * 0.10
    )
    # score ruim (quase aleatório)
    .withColumn(
        "score3",
        F.rand()
    )
)

# Normalizando scores entre 0 e 1
for s in scores:
    min_val = df.agg(F.min(s)).collect()[0][0]
    max_val = df.agg(F.max(s)).collect()[0][0]
    df = df.withColumn(
        s,
        (F.col(s) - F.lit(min_val)) / (F.lit(max_val - min_val))
    )

df.select(grupos + scores + ["target"]).show(5)

+-------+--------+-----------+-----------+-------------------+-------------------+-------------------+------+
|  safra|proposta|flag_grupo1|flag_grupo2|             score1|             score2|             score3|target|
+-------+--------+-----------+-----------+-------------------+-------------------+-------------------+------+
|2023-02|    loja|          1|          0| 0.6169928307026675|0.40323040052300524|0.10583963964753484|     1|
|2023-03|    loja|          1|          1| 0.5703198408902219| 0.4967265152271655| 0.2968537715262256|     0|
|2023-04|parceiro|          0|          1| 0.7770295819733216| 0.7518951377438488| 0.6679358619339034|     0|
|2023-02| digital|          0|          0| 0.4024036853962936|0.43256109356566985| 0.8808335603355195|     0|
|2023-01|    loja|          0|          0|0.36548236756815833| 0.3578485366893646|0.29931492403628684|     0|
+-------+--------+-----------+-----------+-------------------+-------------------+-------------------+------+
only showi

In [3]:
df.count()

20000